In [1]:
import torch
import torch.nn.functional as F

words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [2]:
chars = sorted(list(set(''.join(words))))
# 2. 建立字符到整数的映射 (s_to_i) 编码
s_to_i = {s:i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
# 3. 建立整数到字符的反向映射 (i_to_s) 解码
i_to_s = {i: s for s, i in s_to_i.items()}
print(i_to_s)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
block_size = 3 # 用3个字符来预测下一个词
X, Y = [] , []
for w in words[:5]:
    print(w)
    # context是index形式
    context = [0] * block_size
    for ch in w + '.':
        ix = s_to_i[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(i_to_s[i] for i in context), '---->', i_to_s[ix])
        # 更新index
        context = context[1:] + [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)
print(X) # (32,2)
print(Y)


emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .
tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,

In [ ]:
C = torch.randn((27,2)) # 将27个字符列成2个维度，随机初始化
# 输出的形状 = 索引张量 X 的形状 + 被索引张量 C 剥掉第 0 维后的形状
# 把X里面的数字索引成两维的向量，对着C查表的意思
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [ ]:
W1 = torch.randn((6,100)) # 特征信息 3x2
b1 = torch.randn(100)

In [11]:
torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape

torch.Size([32, 6])

In [12]:
torch.cat(torch.unbind(emb,1),1).shape

torch.Size([32, 6])

In [ ]:
# 最简单的方法，view()
emb.view(32,6)

In [ ]:
# 也就得到了隐藏层 h = w * x +  b
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)